# 🤖 Devi · Creemos el asistente de DevSenior Code (versión de clase)

**Clase 2 · Unidad 2: NLP y el auge de la IA generativa**

En la primera parte de la clase entrenamos **MiniGPT** con 93 diálogos: aprendió el formato de chat, pero sabe muy poco.
Ahora damos el paso que hacen las empresas: tomamos **el mismo modelo** (GPT-2 small en español, 124 M parámetros) y lo ajustamos con
1. **instrucciones generales en español** (una muestra de Alpaca),
2. **nuestros diálogos de clase**, y
3. **la base de conocimiento de DevSenior Code y del Máster**,

para obtener **Devi**, la asistente virtual de la academia.

| | MiniGPT (primera parte) | **Devi (este notebook)** |
|---|---|---|
| Datos | 93 diálogos | muestra de Alpaca + 93 diálogos + **conocimiento DevSenior** |
| Entrenamiento | unos minutos | **5–15 min en GPU** (~10 min en CPU, con datos reducidos) |
| Lo que aprende | el formato de chat | seguir instrucciones + **responder sobre la empresa** + decir "no sé" en precios y fechas |

▶️ **Cómo trabajar:** ejecuta las celdas en orden. Las celdas marcadas con `✍️ LIVE CODING` las escribimos juntos en clase.

## 0. Configuración

Todo lo que cambia entre **GPU** y **CPU** está aquí arriba. En CPU usamos muchos menos datos y congelamos la mitad del modelo: así el entrenamiento cabe en el tiempo de la clase.

In [ ]:
import json, math, random, re, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 140)

MODELO_BASE = "datificate/gpt2-small-spanish"
DATASET_HF = "bertin-project/alpaca-spanish"
RUTA_DIALOGOS = Path("..") / "datasets" / "dialogos_minigpt.jsonl"
RUTA_EMPRESA = Path("..") / "datasets" / "devsenior_conocimiento.jsonl"
DIR_SALIDA = Path("..") / "modelo" / "devi-clase"
NOMBRE_ASISTENTE = "Devi"
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = device.type == "cuda"
USAR_BF16 = GPU and torch.cuda.is_bf16_supported()

# ---------------- CONFIGURACIÓN (GPU vs CPU) ----------------
N_ALPACA         = 6000 if GPU else 300   # muestra de instrucciones generales
REPETIR_EMPRESA  = 10   if GPU else 3     # la base de conocimiento se repite para "grabarla"
REPETIR_DIALOGOS = 3    if GPU else 1
EPOCHS_MAX       = 4    if GPU else 1
MINUTOS_MAX      = 15   if GPU else 10    # presupuesto de tiempo de la clase
MAX_LEN          = 192  if GPU else 128
BATCH_SIZE       = 16   if GPU else 8
CAPAS_CONGELADAS = 0    if GPU else 6     # en CPU solo entrenamos la mitad superior
LEARNING_RATE    = 1e-4 if GPU else 3e-4
EVAL_CADA        = 100  if GPU else 20    # pasos entre evaluaciones
PACIENCIA        = 4

SYSTEM_PROMPT = (f"Eres {NOMBRE_ASISTENTE}, la asistente virtual de DevSenior Code. Respondes en español, "
                 "de forma breve y clara, sobre la academia, el Máster en IA y temas de inteligencia artificial.")
TAG_USUARIO, TAG_BOT = "### Usuario:", f"### {NOMBRE_ASISTENTE}:"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"torch {torch.__version__} | device: {device} | bfloat16: {USAR_BF16}")
if GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("💻 Modo CPU: datos reducidos y capas congeladas para que entrene en ~10 min.")

## 1. Los datos: tres fuentes, un solo formato
Cada ejemplo será un par **pregunta → respuesta**. Las tres fuentes enseñan cosas distintas:
- **Alpaca en español**: seguir instrucciones generales ("escribe", "explica", "resume").
- **Nuestros diálogos**: la personalidad y conceptos de IA (cambiamos "MiniGPT" por "Devi").
- **`devsenior_conocimiento.jsonl`**: 70 temas de la empresa y del Máster, cada uno con varias formas de preguntarlo.

In [ ]:
# ✍️ LIVE CODING — carga devsenior_conocimiento.jsonl, reemplaza {NOMBRE} y agrega la variante sin signos
# 1.1 Base de conocimiento de la empresa
# Reemplazamos {NOMBRE} por el nombre del asistente y creamos una variante sin signos y en minúsculas



In [ ]:
# ✍️ LIVE CODING — carga Alpaca, toma N_ALPACA filas y conviértelas a {pregunta, respuesta}
# 1.2 Una muestra de Alpaca en español (instrucción + input opcional → pregunta)



In [ ]:
# ✍️ LIVE CODING — une alpaca + empresa×REPETIR_EMPRESA + diálogos×REPETIR_DIALOGOS, baraja y separa validación
# 1.3 Unimos las tres fuentes y separamos validación
# Quitamos diálogos de clase que repiten una pregunta de la empresa (evita respuestas contradictorias)



## 2. Formato de chat y máscara −100
Igual que con MiniGPT: el modelo ve `system prompt + ### Usuario: … + ### Devi:` y **solo aprende a escribir la respuesta**.
Los tokens del prompt llevan label `−100`, así no cuentan en la pérdida.

In [ ]:
# ✍️ LIVE CODING — tokenizer, construir_prompt() y codificar() con labels −100 en el prompt
# 2.1 Tokenizador y función que convierte un par pregunta/respuesta en (ids, labels)



In [ ]:
# ✍️ LIVE CODING — aplica codificar() a train y val
# 2.2 Tokenizamos todo y descartamos ejemplos sin respuesta (el prompt ocupó todo MAX_LEN)



## 3. Batches agrupados por largo
Ponemos juntos ejemplos de **largo parecido**: casi no hay relleno (*padding*) y el entrenamiento va más rápido.

In [ ]:
# Batches por largo + collate (rellena con pad y −100). Esta celda viene lista.
def batches_de_epoca(datos, epoca):
    rng = random.Random(SEED + epoca)
    idx = list(range(len(datos))); rng.shuffle(idx)
    lotes = []
    for i in range(0, len(idx), BATCH_SIZE * 20):
        grupo = sorted(idx[i:i + BATCH_SIZE * 20], key=lambda k: len(datos[k][0]))
        lotes += [grupo[j:j + BATCH_SIZE] for j in range(0, len(grupo), BATCH_SIZE)]
    rng.shuffle(lotes)
    return lotes

def collate(datos, indices):
    largo = max(len(datos[i][0]) for i in indices)
    ids = torch.full((len(indices), largo), tokenizer.pad_token_id)
    labels = torch.full((len(indices), largo), -100)
    mask = torch.zeros((len(indices), largo), dtype=torch.long)
    for k, i in enumerate(indices):
        a, b = datos[i]
        ids[k, :len(a)] = torch.tensor(a); labels[k, :len(b)] = torch.tensor(b); mask[k, :len(a)] = 1
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device), "labels": labels.to(device)}

pasos_por_epoca = len(batches_de_epoca(datos_train, 0))
print(f"{pasos_por_epoca} pasos por época · máx {pasos_por_epoca * EPOCHS_MAX} pasos o {MINUTOS_MAX} min")

## 4. Entrenamiento con presupuesto de tiempo
El bucle se detiene por lo primero que ocurra:
- se cumplen **MINUTOS_MAX**,
- se completan **EPOCHS_MAX**, o
- **early stopping**: la validación no mejora en `PACIENCIA` evaluaciones.

Cada vez que la validación mejora guardamos una copia del modelo (⭐) y al final nos quedamos con **la mejor**.

In [ ]:
# ✍️ LIVE CODING — carga el modelo, congela CAPAS_CONGELADAS capas, crea AdamW y el scheduler coseno
# 4.1 Modelo, capas congeladas (en CPU), optimizador y scheduler



In [ ]:
# ✍️ LIVE CODING — evaluar(): pérdida media en validación sin gradientes
# 4.2 Función de evaluación (pérdida media en validación)



In [ ]:
# ✍️ LIVE CODING — bucle con autocast, clip_grad, evaluación cada EVAL_CADA pasos, early stopping y límite de minutos
# 4.3 Bucle de entrenamiento: tiempo máximo + épocas + early stopping, guardando el mejor estado



In [ ]:
# Curvas de pérdida
fig, ax = plt.subplots(figsize=(9, 4))
x, y = zip(*hist_train)
ax.plot(x, pd.Series(y).rolling(20, min_periods=1).mean(), color="#7B2CBF", label="train (suavizado)")
x, y = zip(*hist_val)
ax.plot(x, y, "s--", color="#E0218A", label="validación")
ax.set_xlabel("paso"); ax.set_ylabel("pérdida"); ax.set_title(f"{NOMBRE_ASISTENTE} · entrenamiento de clase")
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

DIR_SALIDA.mkdir(parents=True, exist_ok=True)
modelo.save_pretrained(DIR_SALIDA); tokenizer.save_pretrained(DIR_SALIDA)
print("Modelo guardado en", DIR_SALIDA.resolve())

## 5. ¡Hablemos con Devi!
Generamos en modo **greedy** (siempre el token más probable) para que los datos de la empresa salgan estables, y cortamos cuando el modelo intenta escribir otro `###`.

In [ ]:
# ✍️ LIVE CODING — responder(): tokeniza el prompt, generate greedy y corta en ###
# 5.1 Función responder(): prompt → generate → cortar en "###"



In [ ]:
# ✍️ LIVE CODING — pregúntale a Devi cosas de la empresa, redactadas distinto, que no debe saber y generales
# 5.2 Batería de preguntas: de la empresa, redactadas distinto, "no sé" y generales



## 6. ¿Cuánto memorizó de la empresa?
Para cada tema hacemos su **primera pregunta** y medimos qué porcentaje de las palabras de la respuesta oficial aparece en la respuesta de Devi. Es una métrica simple (no entiende sinónimos), pero sirve para comparar.

In [ ]:
# ✍️ LIVE CODING — para cada tema, compara las palabras de la respuesta oficial con las de Devi
# 6.1 Cobertura de palabras por tema



In [ ]:
# 6.2 Comparación: Devi de clase vs Devi entrenado ~2 h (si el docente lo dejó en ../modelo/devi-devsenior)
ruta_largo = Path("..") / "modelo" / "devi-devsenior"
if ruta_largo.exists():
    largo = AutoModelForCausalLM.from_pretrained(ruta_largo).to(device).eval()
    comp = pd.DataFrame([{"pregunta": p, "Devi clase": responder(modelo, p),
                          "Devi largo (~2 h)": responder(largo, p)}
                         for lista in preguntas.values() for p in lista])
    m_largo = medir_memoria(largo)
    print(f"Memoria (temas ≥ 70 %): clase {(memoria.cobertura >= .7).mean():.0%} · largo {(m_largo.cobertura >= .7).mean():.0%}")
    display(comp)
else:
    print("No está el Devi largo en", ruta_largo, "→ se omite la comparación.")

In [ ]:
# Chat libre con Devi (escribe 'salir' para terminar)
def chatear(muestrear=False):
    while True:
        p = input("🧑 Tú: ").strip()
        if p.lower() in {"salir", ""}:
            break
        print(f"🤖 {NOMBRE_ASISTENTE}:", responder(modelo, p, muestrear=muestrear), "\n")

# chatear()

## 7. 🏁 Reto
1. **Agrega un tema nuevo** a la base de conocimiento (por ejemplo, el horario de atención de tu sede ficticia) con **3 formas de preguntarlo**, vuelve a entrenar y verifica si Devi lo responde.
2. Pregúntale lo mismo **con palabras muy distintas** a las del dataset. ¿Acierta?
3. **Discusión:** si mañana cambia el teléfono de contacto, ¿qué harías con Devi? ¿Volver a entrenar o buscar otra solución?

> 💡 Pista para el punto 3: busca qué es **RAG (Retrieval-Augmented Generation)**. Lo veremos más adelante.

In [ ]:
# ✍️ LIVE CODING — crea 3 filas {id, pregunta, respuesta} y agrégalas a `empresa`
# Reto 1: agrega tu tema aquí y vuelve a ejecutar desde la sección 1.3

